In [1]:
from helpers.models import Models
from helpers.llm_client import LLMClient
from helpers.functions import *
from helpers.parsers import license_parser
import pandas as pd
import os
import nirjas
from sklearn.metrics import accuracy_score

pd.set_option('display.max_rows', None)    # Show all rows
# pd.set_option('display.max_colwidth', None)  # Show full column width

/home/jimbo/Desktop/GSoC24/repo/GSoC24/gsoc24env/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/home/jimbo/Desktop/GSoC24/repo/GSoC24/gsoc24env/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
# Check to make sure that all API keys are present
os.environ['GROQ_API_KEY'] 
os.environ['NVIDIA_API_KEY']
os.environ['TOGETHER_API_KEY']    
'OK'
#

'OK'

In [2]:
# Create the dataset of licenses/obligations
license_text_path = 'extras/license_information/details'
license_text_files = os.listdir(license_text_path)

obligations_text_path = 'extras/obligations'
obligations_text_files = os.listdir(obligations_text_path)

df = pd.DataFrame(columns=['License Name', 'License ID', 'License Text', 'Obligations'])

for license_file in os.listdir(license_text_path):
    base_filename = os.path.splitext(license_file)[0]

    obligation_filename = base_filename + ".txt"
    obligation_filepath = os.path.join(obligations_text_path, obligation_filename)

    if os.path.exists(obligation_filepath):
        license = read_json_file(os.path.join(license_text_path, license_file))
        with open(obligation_filepath, 'r') as f:
            obligations = f.read()
        df.loc[len(df)] = [license['name'], base_filename, license['licenseText'], obligations] 

print(f'Number of Licenses found with Obligations: {len(df)}')
print(f'Number of License Files: {len(license_text_files)}')
print(f'Number of Obligation Files: {len(obligations_text_files)}')

FileNotFoundError: [Errno 2] No such file or directory: 'extras/license_information/details'

In [4]:
# Obligations with no license files
license_filenames = set(os.path.splitext(file)[0] for file in os.listdir(license_text_path))
obligation_filenames = set(os.path.splitext(file)[0] for file in os.listdir(obligations_text_path))

mismatched_filenames = obligation_filenames - license_filenames

mismatched_obligation_files = [filename + ".txt" for filename in mismatched_filenames]
mismatched_obligation_files

['LicenseRef-scancode-ppp.txt',
 'LicenseRef-scancode-info-zip-2003-05.txt',
 'GPL-2.0-only WITH Classpath-exception-2.0.txt']

In [121]:
print("THE BEER-WARE LICENSE\" (Revision 42):  \u003cphk@FreeBSD.ORG\u003e wrote this file. As long as you retain this notice you  can do whatever you want with this stuff. If we meet some day, and you think  this stuff is worth it, you can buy me a beer in return Poul-Henning Kamp\n")

THE BEER-WARE LICENSE" (Revision 42):  <phk@FreeBSD.ORG> wrote this file. As long as you retain this notice you  can do whatever you want with this stuff. If we meet some day, and you think  this stuff is worth it, you can buy me a beer in return Poul-Henning Kamp



In [119]:
license_filenames - obligation_filenames

{'3D-Slicer-1.0',
 'AAL',
 'ADSL',
 'AFL-1.1',
 'AFL-1.2',
 'AGPL-1.0',
 'AGPL-1.0-only',
 'AGPL-1.0-or-later',
 'AGPL-3.0',
 'AMD-newlib',
 'AMDPLPA',
 'AML',
 'AML-glslang',
 'AMPAS',
 'ANTLR-PD',
 'ANTLR-PD-fallback',
 'APAFML',
 'APL-1.0',
 'APSL-1.0',
 'APSL-1.1',
 'APSL-1.2',
 'APSL-2.0',
 'ASWF-Digital-Assets-1.0',
 'ASWF-Digital-Assets-1.1',
 'Abstyles',
 'AdaCore-doc',
 'Adobe-2006',
 'Adobe-Display-PostScript',
 'Adobe-Glyph',
 'Adobe-Utopia',
 'Afmparse',
 'Aladdin',
 'App-s2p',
 'Arphic-1999',
 'Artistic-1.0-cl8',
 'BSD-2-Clause-Darwin',
 'BSD-2-Clause-FreeBSD',
 'BSD-2-Clause-NetBSD',
 'BSD-2-Clause-Views',
 'BSD-2-Clause-first-lines',
 'BSD-3-Clause-Attribution',
 'BSD-3-Clause-Clear',
 'BSD-3-Clause-HP',
 'BSD-3-Clause-LBNL',
 'BSD-3-Clause-Modification',
 'BSD-3-Clause-No-Military-License',
 'BSD-3-Clause-No-Nuclear-License',
 'BSD-3-Clause-No-Nuclear-License-2014',
 'BSD-3-Clause-No-Nuclear-Warranty',
 'BSD-3-Clause-Sun',
 'BSD-3-Clause-acpica',
 'BSD-3-Clause-flex',
 

In [5]:
def prompt_1(license_text, obligations):
    return f"""
    You are a legal expert specializing in open-source software licenses. Your task is to verify the accuracy of a set of obligations against a given open-source license text. This means you will determine if each obligation statement is supported by, contradicts, or is not explicitly mentioned in the license.

    **Context:**

    Open-source licenses grant users certain rights and impose specific obligations when using or distributing software. These obligations are crucial for ensuring compliance and avoiding legal issues.

    **Guidelines:**

    1.  **Thorough Analysis:** Carefully read and understand the provided open-source license text. Pay attention to key terms, conditions, permissions, and restrictions.
    2.  **Break Down Obligations:** Divide the provided obligations into individual clauses or statements.
    3.  **Verify Against License:** For each obligation clause, compare it against the relevant sections of the open-source license text.
    4.  **Categorize Validity:** Assess the validity of each obligation clause:
        *   **Valid:** The obligation accurately reflects a requirement in the license.
        *   **Invalid:** The obligation contradicts or is not supported by the license.
        *   **Partially Valid:** The obligation is partially accurate or open to interpretation based on the license.
    5.  **Explain Your Reasoning:** Provide a clear explanation for each validity assessment. Cite specific sections of the open-source license text and explain any interpretation or judgment calls.
    6.  **Consider Implicit Obligations:** Some obligations may not be explicitly stated but can be inferred from the overall intent of the license. If you identify such implicit obligations, explain your reasoning based on the specific license text.
    7.  **Handle Conditional Obligations:** If an obligation depends on certain conditions (e.g., "if used for commercial purposes"), assess its validity under those conditions according to the license text.
    8.  **Flag Ambiguities:** If any part of the license or obligation is unclear, flag it and explain the potential interpretations based on the license text.

    **Output Format:**

    Present your analysis in a table format:

    | Obligation Clause                                   | Validity    | Explanation                                                                                                                                                                                             |
    | --------------------------------------------------- | ----------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
    | [Obligation Clause 1]                               | [Validity]  | [Your detailed explanation, citing specific sections of the license text and any relevant interpretations]                                                                               |
    | [Obligation Clause 2]                               | [Validity]  | [Your detailed explanation]                                                                                                                                                                        |
    | ...                                                 | ...         | ...                                                                                                                                                                                                |

    **Additional Notes:**

    *   If a single obligation clause contains multiple parts connected by "OR" or "AND," assess each part separately and then as a whole.
    *   Be concise but thorough in your explanations.
    *   Use clear, unambiguous language.
    *   Maintain a professional, objective tone.

    **License Text:**
    {license_text}

    **Obligations:**
    {obligations}
    """

In [6]:
def prompt_2(license_text, obligations):
    return f"""
        [Task]
        Evaluate the accuracy of each clause within the provided set of obligations against the given open-source license text. Determine if each clause is valid (supported by the license), invalid (contradicts the license), or partially valid (partially accurate or open to interpretation).

        [Instructions]
            1. Carefully analyze the open-source license text.
            2. Examine each clause within the provided obligations.
            3. Compare each clause to the relevant sections of the license text.
            4. Categorize each clause as valid, invalid, or partially valid.
            5. Provide a clear explanation for each assessment, citing specific license text sections and any interpretations.
            6. Present your analysis in the following list format only, without any additional text or commentary:
                Clause: [Clause text]
                Result: [valid/invalid/partially valid]
                Explanation: [Your detailed explanation, citing specific license text sections and any relevant interpretations]
            7. Be Concise and only follow the output format provided in Instruction 6 without any introductions or conclusions
        
        [Additional Notes]
            1. if a clause contains multiple parts, assess each part separately and then as a whole.
            2. Be concise yet thorough in your explanations.
            3. Use clear, unambiguous language.
            4. Maintain a professional, objective tone.
        
        [License Text]
        {license_text}

        [Corresponding Obligations]
        {obligations}
    """

In [103]:
def convert(x):
    return f"""Convert the Following License Text to License Obligations
    License Text:
    {x}
"""


df_prompt = df.copy(deep=True)
df_prompt['Prompt'] = df_prompt['License Text'].apply(convert)
df_prompt_filtered = df_prompt[
    (df_prompt['Prompt'].astype(str).apply(len) < 40000) &
    (df_prompt['Obligations'].astype(str).apply(len) < 5000)
]

# Inform about removed rows (optional)
removed_rows = len(df_prompt) - len(df_prompt_filtered)
if removed_rows > 0:
    print(f"Removed {removed_rows} rows exceeding character limits.")

# Save filtered results
df_prompt_filtered[['Obligations', 'Prompt']].to_csv('test_1.csv')


Removed 10 rows exceeding character limits.


In [7]:
client = LLMClient()
df = client.process_dataset_obligations(df, Models.LLAMA_3_8b, prompt_2, 'obligation_validation', log_every = 5, retry_fails=False)

2024-08-01 10:11:07.585 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 0
2024-08-01 10:11:15.812 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 5
2024-08-01 10:11:34.641 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 10
2024-08-01 10:12:17.309 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 15
2024-08-01 10:12:28.585 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 20
2024-08-01 10:13:11.470 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 25
2024-08-01 10:13:35.540 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 30
2024-08-01 10:13:56.177 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 35
2024-08-01 10:14:24.541 | INFO     | helpers.llm_client:process_dataset_obligations:319 - Processing index: 40
202

In [9]:
df_remaining = df[df['response'].notna()]

In [79]:
def parse_license_clauses(input_text):
    """Parses license clauses, results, and explanations from the input text."""
    clauses = []
    current_clause = {}
    expecting_result = False
    expecting_explanation = False

    input_text = input_text.strip()

    for line in input_text.splitlines():
        if "Clause" in line and not expecting_result and not expecting_explanation:
            # Start of a new clause
            if current_clause:
                clauses.append(current_clause)
            current_clause = {"Clause": line.strip()}
            expecting_result = True
        elif expecting_result:
            current_clause["Result"] = line.strip()
            expecting_result = False
            expecting_explanation = True
        elif expecting_explanation:
            current_clause["Explanation"] = line.strip()
            expecting_explanation = False

    # Add the last clause if there is one
    if current_clause:
        clauses.append(current_clause)

    return clauses

def calculate_row_score(parsed_clauses):
    """Calculates the overall score for a row of parsed clauses."""

    try:
        scores = {"valid": 1, "partially valid": 0.5, "invalid": 0}
        total_expected = len(parsed_clauses)
        total_score = 0

        for clause in parsed_clauses:
            result_lower = clause["Result"].lower()
            if any(key in result_lower for key in scores.keys()):
                total_score += scores[max(scores, key=lambda k: scores[k] if k in result_lower else 0)]  
            else:  # Default to 0 if no recognized result is found
                total_score += 0

        return total_score / total_expected
    except:
        return None

def process_license_dataframe(df):
    """Processes the DataFrame with license texts and calculates scores."""

    df["Parsed Clauses"] = df["response"].apply(parse_license_clauses)
    df["Parsed Clauses"] = df["Parsed Clauses"].apply(lambda x: x if x else None)
    df.dropna(subset=["Parsed Clauses"], inplace=True)
    df["Overall Score"] = df["Parsed Clauses"].apply(calculate_row_score)
    overall_accuracy = np.nanmean(df["Overall Score"]) 
    return df, overall_accuracy

In [81]:
df_remaining, overall_accuracy = process_license_dataframe(df_remaining)
print(overall_accuracy)

0.9274106343726598


/tmp/ipykernel_36465/1935400301.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Parsed Clauses"] = df["response"].apply(parse_license_clauses)
/tmp/ipykernel_36465/1935400301.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Parsed Clauses"] = df["Parsed Clauses"].apply(lambda x: x if x else None)
/tmp/ipykernel_36465/1935400301.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pa

In [109]:
print(df_remaining_2.loc[3, 'Obligations'])

USE CASE Source code delivery OR Binary delivery
	YOU MUST Provide License text
		ATTRIBUTE Viewable
	YOU MUST Provide Legal notices
	IF NOT Legal notices
		YOU MUST Provide Copyright notice "W3C Software and Document Short Notice https://www.w3.org/Consortium/Legal/2015/copyright-software-short-notice.html"
	IF Software modification
		YOU MUST Provide Copyright notice AND Modification notice
	YOU MUST NOT Promote



In [110]:
print(df_remaining_2.loc[3, 'response'])

Here is the evaluation of each clause against the given open-source license text:

Clause: YOU MUST Provide License text
Result: Valid
Explanation: The license text explicitly states that the licensee agrees to comply with the terms and conditions, which includes providing the full text of the NOTICE (license text) on ALL copies of the work or portions thereof, including modifications. (Section: Permission to copy, modify, and distribute this work)

Clause: ATTRIBUTE Viewable
Result: Partially Valid
Explanation: The license text does not explicitly mention the requirement to attribute the work as "Viewable". However, it does require the inclusion of the full text of the NOTICE (license text) on ALL copies of the work or portions thereof, including modifications. This implies that the attribution should be viewable, but the exact wording is not specified. (Section: Permission to copy, modify, and distribute this work)

Clause: YOU MUST Provide Legal notices
Result: Valid
Explanation: Th

In [86]:
df_remaining_2 = df_remaining[df_remaining['Overall Score'].notna()]
df_remaining_2[df_remaining_2['Overall Score'] != 1.0]

,License Name,License ID,License Text,Obligations,response,Parsed Clauses,Overall Score
12,Mozilla Public License 1.1,MPL-1.1,Mozilla Public License Version 1.1\n\n1. Defin...,USE CASE Source code delivery\n\tYOU MUST Prov...,Here is the evaluation of each clause against ...,"[{'Clause': '**Clause 1: Definitions**', 'Resu...",0.000000
13,bzip2 and libbzip2 License v1.0.6,bzip2-1.0.6,"This program, ""bzip2"", the associated library ...",USE CASE Source code delivery\n\tYOU MUST Forw...,Here is the evaluation of each clause against ...,[{'Clause': 'Clause: YOU MUST Forward Copyrigh...,0.444444
34,Apache License 2.0,Apache-2.0,"Apache License\nVersion 2.0, January 2004\nhtt...",USE CASE Source code delivery\n\tYOU MUST Prov...,Here is the evaluation of each clause within t...,[{'Clause': 'Explanation: Clause 4(a) of the A...,0.000000
39,RSA Message-Digest License,RSA-MD,"Copyright (C) 1991-2, RSA Data Security, Inc. ...",USE CASE Source code delivery OR Binary delive...,Here is the evaluation of each clause against ...,[{'Clause': 'Clause: USE CASE Source code deli...,0.666667
51,zlib License,Zlib,zlib License\n\nThis software is provided 'as-...,USE CASE Source code delivery\n\tYOU MUST Forw...,Here is the evaluation of each clause against ...,[{'Clause': 'Clause: YOU MUST Forward License ...,0.600000
71,W3C Software Notice and License (2002-12-31),W3C,W3C SOFTWARE NOTICE AND LICENSE\n\nThis work (...,USE CASE Source code delivery OR Binary delive...,Here is the evaluation of each clause within t...,[{'Clause': 'Clause: YOU MUST Provide License ...,0.500000
76,BSD Source Code Attribution,BSD-Source-Code,"Copyright (c) 2011, Deusty, LLC\nAll rights re...",USE CASE Source code delivery\n\tYOU MUST Forw...,Here is the evaluation of each clause against ...,[{'Clause': 'Clause: YOU MUST Forward Copyrigh...,0.800000
80,zlib/libpng License with Acknowledgement,zlib-acknowledgement,Copyright (c) 2002-2007 Charlie Poole\nCopyrig...,USE CASE Source code delivery\n\tYOU MUST Forw...,Here is the evaluation of each clause against ...,[{'Clause': 'Clause: YOU MUST Forward License ...,0.428571
91,SSH short notice,SSH-short,"As far as I am concerned, the code I have writ...",USE CASE Source code delivery OR Binary delive...,Here is the analysis of the obligations agains...,"[{'Clause': 'Clause: As far as I am concerned,...",0.750000
96,BSD-4-Clause (University of California-Specific),BSD-4-Clause-UC,BSD-4-Clause (University of California-Specifi...,USE CASE Source code delivery\n\tYOU MUST Forw...,Here is the evaluation of each clause against ...,[{'Clause': 'Here is the evaluation of each cl...,0.909091


In [117]:
print(df_remaining_2.loc[12	, 'Obligations'])

USE CASE Source code delivery
	YOU MUST Provide Standard license notice (Exhibit A)
	YOU MUST Provide License text
	IF Documentation
		YOU MUST Provide License text
	IF Patent holder OR Trademark holder OR Third-party patents OR Third-party trademarks
		YOU MUST Provide File "LEGAL"
			ATTRIBUTE Crediting Patent holder AND Trademark holder AND Third-party patents AND Third-party trademarks
			IF ATTRIBUTE Dynamic
				YOU MUST Update File "LEGAL"
				YOU MUST Disseminate Patent notice AND Trademark notice
	IF Software modification
		YOU MUST Grant License
			ATTRIBUTE Original license
		YOU MUST Provide Modification report
			ATTRIBUTE Documentation of Software modifications
			ATTRIBUTE Modification date
		YOU MUST Provide Modification notice
			ATTRIBUTE Highlighted
			ATTRIBUTE Crediting Initial developer
			ATTRIBUTE Included In Source code
	YOU MUST NOT Restrict Granted rights
	IF Service offerings
		YOU MUST Indemnify Other contributors
USE CASE Binary delivery
	YOU MUST Include N

In [116]:
print(df_remaining_2.loc[12	, 'License Text'])

Mozilla Public License Version 1.1

1. Definitions.

     1.0.1. "Commercial Use" means distribution or otherwise making the Covered Code available to a third party.

     1.1. "Contributor" means each entity that creates or contributes to the creation of Modifications.

     1.2. "Contributor Version" means the combination of the Original Code, prior Modifications used by a Contributor, and the Modifications made by that particular Contributor.

     1.3. "Covered Code" means the Original Code or Modifications or the combination of the Original Code and Modifications, in each case including portions thereof.

     1.4. "Electronic Distribution Mechanism" means a mechanism generally accepted in the software development community for the electronic transfer of data.

     1.5. "Executable" means Covered Code in any form other than Source Code.

     1.6. "Initial Developer" means the individual or entity identified as the Initial Developer in the Source Code notice required by Exhibit A

In [118]:
print(df_remaining_2.loc[12, 'response'])

Here is the evaluation of each clause against the Mozilla Public License Version 1.1:

**Clause 1: Definitions**

* Result: Valid
* Explanation: The definitions provided in this clause are clear and concise, and they establish the foundation for the rest of the license.

**Clause 2: Source Code License**

* Result: Valid
* Explanation: This clause grants the recipient a license to use, reproduce, modify, display, perform, sublicense, and distribute the Covered Code, which is in line with the terms of the license.

**Clause 3: Distribution Obligations**

* Result: Valid
* Explanation: This clause requires the distributor to include the notice in Exhibit A, provide the license text, and make the Source Code available under the terms of the license.

**Clause 4: Inability to Comply Due to Statute or Regulation**

* Result: Valid
* Explanation: This clause provides a mechanism for the distributor to comply with laws and regulations that may prohibit certain actions, while still allowing th